# Teste XGBoost - Do Zero vs Biblioteca Pronta

Este notebook tem um objetivo didático: entender o **XGBoost** de duas formas complementares.

- **Seção 1 - Na mão (do zero):** vamos implementar o algoritmo passo a passo, apenas com `numpy`, seguindo a matemática do Gradient Boosting (gradientes, hessianos, ganho de split, valor de folha). Usamos um exemplo minúsculo de classificação binária para conseguir acompanhar cada conta.
- **Seção 2 - Com a lib pronta:** o mesmo exemplo, agora com `import xgboost as xgb`, para ver como a biblioteca faz tudo isso por baixo dos panos e confirmar que os resultados batem.

> A matemática usada aqui segue exatamente o documento `docs/CRISP-DM/MODELAGEM_ML/XGBOOST_CONCEITO.MD` do projeto.

---
# Seção 1 - XGBoost do Zero (só com numpy)

A ideia central do XGBoost para classificação binária é:

1. Começar com um **palpite inicial** igual para todo mundo (o *base score*).
2. Medir o **erro** de cada observação usando a derivada da função de perda (gradiente `g` e hessiano `h`).
3. Treinar uma **pequena árvore** que aprende a prever esses gradientes.
4. **Somar** essa árvore ao modelo, com um peso pequeno (learning rate).
5. Repetir várias vezes - cada árvore corrige o que sobrou da anterior.

Vamos construir isso peça por peça.

## 1.1 - Importações e o nosso mini-dataset

Para conseguir acompanhar a matemática na mão, usamos só **6 funcionários** e **2 features**:

- `anos_empresa`: há quantos anos a pessoa está na empresa.
- `satisfacao`: nota de satisfação de 1 a 4.

O alvo `saiu` é binário: `1` = o funcionário saiu (turnover), `0` = ficou.

A intuição do exemplo: quem tem **pouca satisfação** e **pouco tempo de casa** tende a sair.

In [1]:
import numpy as np

# Mini-dataset: 6 funcionarios, 2 features
# Colunas: [anos_empresa, satisfacao]
X = np.array([
    [1, 1],   # pouco tempo, pouca satisfacao  -> saiu
    [2, 1],   # pouco tempo, pouca satisfacao  -> saiu
    [1, 2],   # pouco tempo, satisfacao media  -> saiu
    [5, 3],   # bastante tempo, satisfacao boa -> ficou
    [6, 4],   # bastante tempo, satisfacao alta-> ficou
    [8, 3],   # bastante tempo, satisfacao boa -> ficou
], dtype=float)

y = np.array([1, 1, 1, 0, 0, 0], dtype=float)  # 1 = saiu, 0 = ficou

feature_names = ['anos_empresa', 'satisfacao']

print('X (features):')
print(X)
print('\ny (alvo - saiu?):', y)
print('Taxa de turnover:', y.mean())

X (features):
[[1. 1.]
 [2. 1.]
 [1. 2.]
 [5. 3.]
 [6. 4.]
 [8. 3.]]

y (alvo - saiu?): [1. 1. 1. 0. 0. 0.]
Taxa de turnover: 0.5


## 1.2 - Etapa 1: Base Score (o palpite inicial)

Antes de qualquer árvore, o XGBoost chuta o mesmo valor para todos. Em classificação binária, esse chute é o **log-odds** da taxa de eventos:

$$ \hat{y}_0 = \log\left(\frac{\bar{y}}{1 - \bar{y}}\right) $$

onde $\bar{y}$ é a proporção de positivos.

**Atenção a uma sutileza importante:** o XGBoost trabalha internamente com o *logit* (o score bruto, que pode ser qualquer número real), e só no final converte para probabilidade usando a **sigmoide**:

$$ p = \sigma(\hat{y}) = \frac{1}{1 + e^{-\hat{y}}} $$

No nosso dataset $\bar{y} = 0.5$, então o log-odds é $\log(0.5/0.5) = \log(1) = 0$. Ou seja: começamos com 50% de probabilidade para todo mundo.

In [4]:
def sigmoid(x):
    """Converte o score bruto (logit) em probabilidade entre 0 e 1."""
    return 1.0 / (1.0 + np.exp(-x))

y_mean = y.mean()
base_score = np.log(y_mean / (1 - y_mean))  # log-odds

print(f'Taxa de evento (y_mean): {y_mean:.4f}')
print(f'Base score (log-odds):   {base_score:.4f}')
print(f'Probabilidade inicial:   {sigmoid(base_score):.4f}')

# Previsao acumulada (em logit) comeca igual para todos
y_pred_logit = np.full(len(y), base_score)
print('\nPrevisao inicial (logit) para cada funcionario:', y_pred_logit)

Taxa de evento (y_mean): 0.5000
Base score (log-odds):   0.0000
Probabilidade inicial:   0.5000

Previsao inicial (logit) para cada funcionario: [0. 0. 0. 0. 0. 0.]


## 1.3 - Etapa 2: Gradiente e Hessiano (medindo o erro)

Para cada observação, calculamos duas derivadas da perda (Binary Cross-Entropy):

$$ g_i = p_i - y_i \qquad\text{(gradiente - direção e tamanho do erro)} $$
$$ h_i = p_i \cdot (1 - p_i) \qquad\text{(hessiano - curvatura/confiança)} $$

onde $p_i = \sigma(\hat{y}_i)$ é a probabilidade prevista atual.

Interpretação prática:
- Se o modelo prevê $p=0.5$ para alguém que **saiu** ($y=1$), então $g = 0.5 - 1 = -0.5$. O sinal negativo diz: "preciso **aumentar** o score dessa pessoa".
- Se prevê $p=0.5$ para alguém que **ficou** ($y=0$), então $g = 0.5 - 0 = +0.5$. "preciso **diminuir** o score".

A árvore que vem a seguir vai aprender justamente a prever esses gradientes.

In [5]:
def grad_hess(y_true, y_pred_logit):
    """Gradiente e hessiano da Binary Cross-Entropy."""
    p = sigmoid(y_pred_logit)
    g = p - y_true        # gradiente
    h = p * (1 - p)       # hessiano
    return g, h

g, h = grad_hess(y, y_pred_logit)

print('Prob. atual (p):', np.round(sigmoid(y_pred_logit), 3))
print('Alvo real  (y):', y)
print('Gradiente  (g):', np.round(g, 3), '  <- negativo p/ quem saiu, positivo p/ quem ficou')
print('Hessiano   (h):', np.round(h, 3), '  <- 0.25 = maxima incerteza (p=0.5)')

Prob. atual (p): [0.5 0.5 0.5 0.5 0.5 0.5]
Alvo real  (y): [1. 1. 1. 0. 0. 0.]
Gradiente  (g): [-0.5 -0.5 -0.5  0.5  0.5  0.5]   <- negativo p/ quem saiu, positivo p/ quem ficou
Hessiano   (h): [0.25 0.25 0.25 0.25 0.25 0.25]   <- 0.25 = maxima incerteza (p=0.5)


## 1.4 - Etapas 3 e 6: Ganho do Split (onde cortar a árvore)

A árvore procura o melhor ponto de corte (feature + threshold). Para cada candidato, calculamos o **ganho**:

$$ \text{Gain} = \frac{1}{2}\left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L+G_R)^2}{H_L+H_R + \lambda} \right] - \gamma $$

onde $G = \sum g_i$ e $H = \sum h_i$ dos lados esquerdo (L) e direito (R), $\lambda$ é a regularização L2 (`reg_lambda`) e $\gamma$ é o ganho mínimo (`gamma`).

Quanto **maior** o ganho, melhor o split separa os gradientes. Testamos todos os cortes possíveis e ficamos com o de maior ganho.

In [6]:
def calc_gain(G_L, H_L, G_R, H_R, lam=1.0, gamma=0.0):
    termo_L = G_L**2 / (H_L + lam)
    termo_R = G_R**2 / (H_R + lam)
    termo_total = (G_L + G_R)**2 / (H_L + H_R + lam)
    return 0.5 * (termo_L + termo_R - termo_total) - gamma


def melhor_split(X, g, h, lam=1.0, gamma=0.0):
    """Varre todas as features e thresholds e retorna o corte de maior ganho."""
    melhor = {'gain': -np.inf, 'feature': None, 'threshold': None}
    G_total, H_total = g.sum(), h.sum()

    for feat in range(X.shape[1]):
        valores = np.unique(X[:, feat])
        # candidatos a threshold = pontos medios entre valores vizinhos
        thresholds = (valores[:-1] + valores[1:]) / 2
        for thr in thresholds:
            mask_L = X[:, feat] <= thr
            G_L, H_L = g[mask_L].sum(), h[mask_L].sum()
            G_R, H_R = G_total - G_L, H_total - H_L
            gain = calc_gain(G_L, H_L, G_R, H_R, lam, gamma)
            if gain > melhor['gain']:
                melhor = {'gain': gain, 'feature': feat, 'threshold': thr}
    return melhor

split = melhor_split(X, g, h, lam=1.0)
print('Melhor split encontrado:')
print(f"  feature  = {feature_names[split['feature']]}")
print(f"  threshold= {split['threshold']}")
print(f"  ganho    = {split['gain']:.4f}")

Melhor split encontrado:
  feature  = anos_empresa
  threshold= 3.5
  ganho    = 1.2857


## 1.5 - Etapa 4: Valor Ótimo da Folha

Uma vez definido o corte, cada folha recebe um valor calculado analiticamente (nada de média simples como na Random Forest):

$$ w^* = -\frac{\sum g_i}{\sum h_i + \lambda} $$

Esse é o ajuste (em logit) que a árvore aplica para quem cai naquela folha. O $\lambda$ no denominador encolhe o valor - regularização.

In [7]:
def valor_folha(g_folha, h_folha, lam=1.0):
    return -g_folha.sum() / (h_folha.sum() + lam)

# Aplicando o split encontrado para separar as duas folhas
feat, thr = split['feature'], split['threshold']
mask_L = X[:, feat] <= thr
mask_R = ~mask_L

w_L = valor_folha(g[mask_L], h[mask_L], lam=1.0)
w_R = valor_folha(g[mask_R], h[mask_R], lam=1.0)

print(f"Corte: {feature_names[feat]} <= {thr}")
print(f"  Folha ESQUERDA (<= {thr}): funcionarios {np.where(mask_L)[0]} -> w* = {w_L:.4f}")
print(f"  Folha DIREITA  (>  {thr}): funcionarios {np.where(mask_R)[0]} -> w* = {w_R:.4f}")
print('\nRepare: a folha com os que SAIRAM tende a w* positivo (empurra a prob. p/ cima)')
print('e a folha com os que FICARAM tende a w* negativo (empurra a prob. p/ baixo).')

Corte: anos_empresa <= 3.5
  Folha ESQUERDA (<= 3.5): funcionarios [0 1 2] -> w* = 0.8571
  Folha DIREITA  (>  3.5): funcionarios [3 4 5] -> w* = -0.8571

Repare: a folha com os que SAIRAM tende a w* positivo (empurra a prob. p/ cima)
e a folha com os que FICARAM tende a w* negativo (empurra a prob. p/ baixo).


## 1.6 - Juntando tudo: uma árvore de 1 corte (stump)

Para manter simples, cada árvore do nosso XGBoost vai ter só **um corte** (um *decision stump*, profundidade 1). Isso já é suficiente para enxergar o boosting funcionando. A classe abaixo encapsula as etapas 3, 4 e 6.

In [8]:
class ArvoreStump:
    """Arvore de 1 corte que aprende a prever os gradientes (XGBoost estilo)."""
    def __init__(self, lam=1.0, gamma=0.0):
        self.lam = lam
        self.gamma = gamma

    def fit(self, X, g, h):
        s = melhor_split(X, g, h, self.lam, self.gamma)
        self.feature = s['feature']
        self.threshold = s['threshold']
        mask_L = X[:, self.feature] <= self.threshold
        self.w_L = valor_folha(g[mask_L], h[mask_L], self.lam)
        self.w_R = valor_folha(g[~mask_L], h[~mask_L], self.lam)
        return self

    def predict(self, X):
        mask_L = X[:, self.feature] <= self.threshold
        return np.where(mask_L, self.w_L, self.w_R)

# Testando uma unica arvore sobre os gradientes atuais
arvore = ArvoreStump(lam=1.0).fit(X, g, h)
print('Ajustes (logit) que a 1a arvore aplica a cada funcionario:')
print(np.round(arvore.predict(X), 4))

Ajustes (logit) que a 1a arvore aplica a cada funcionario:
[ 0.8571  0.8571  0.8571 -0.8571 -0.8571 -0.8571]


## 1.7 - Etapa 5: O loop de boosting completo

Agora juntamos tudo no laço sequencial. A cada iteração:

1. calculamos `g` e `h` com as previsões atuais;
2. treinamos uma árvore nova sobre esses gradientes;
3. somamos a árvore ao modelo com encolhimento (learning rate $\eta$):

$$ \hat{y}_t = \hat{y}_{t-1} + \eta \cdot f_t(x) $$

Vamos acompanhar a perda (log-loss) caindo a cada árvore - é o sinal de que o boosting está corrigindo os erros.

In [9]:
def log_loss(y_true, y_pred_logit):
    p = sigmoid(y_pred_logit)
    p = np.clip(p, 1e-7, 1 - 1e-7)  # evita log(0)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))


class MeuXGBoost:
    def __init__(self, n_arvores=10, learning_rate=0.3, lam=1.0, gamma=0.0):
        self.n_arvores = n_arvores
        self.eta = learning_rate
        self.lam = lam
        self.gamma = gamma
        self.arvores = []

    def fit(self, X, y, verbose=True):
        y_mean = y.mean()
        self.base_score = np.log(y_mean / (1 - y_mean))
        pred_logit = np.full(len(y), self.base_score)

        for t in range(self.n_arvores):
            g, h = grad_hess(y, pred_logit)
            arvore = ArvoreStump(self.lam, self.gamma).fit(X, g, h)
            pred_logit += self.eta * arvore.predict(X)   # Etapa 5
            self.arvores.append(arvore)
            if verbose:
                perda = log_loss(y, pred_logit)
                print(f'Arvore {t+1:2d} | log-loss = {perda:.4f}')
        return self

    def predict_logit(self, X):
        logit = np.full(len(X), self.base_score)
        for arvore in self.arvores:
            logit += self.eta * arvore.predict(X)
        return logit

    def predict_proba(self, X):
        return sigmoid(self.predict_logit(X))

    def predict(self, X, limiar=0.5):
        return (self.predict_proba(X) >= limiar).astype(int)


meu_modelo = MeuXGBoost(n_arvores=10, learning_rate=0.3, lam=1.0).fit(X, y)

Arvore  1 | log-loss = 0.5728
Arvore  2 | log-loss = 0.4805
Arvore  3 | log-loss = 0.4085
Arvore  4 | log-loss = 0.3514
Arvore  5 | log-loss = 0.3055
Arvore  6 | log-loss = 0.2682
Arvore  7 | log-loss = 0.2375
Arvore  8 | log-loss = 0.2120
Arvore  9 | log-loss = 0.1906
Arvore 10 | log-loss = 0.1725


## 1.8 - Resultado do modelo feito do zero

Vamos ver as probabilidades finais. A expectativa: probabilidade **alta** para os três que saíram (índices 0, 1, 2) e **baixa** para os três que ficaram (índices 3, 4, 5).

In [10]:
proba = meu_modelo.predict_proba(X)
pred = meu_modelo.predict(X)

print('Funcionario | anos | satisf | real | prob(saiu) | previsto')
print('-' * 60)
for i in range(len(X)):
    print(f'     {i}      |  {int(X[i,0])}   |   {int(X[i,1])}    |  {int(y[i])}   |   {proba[i]:.3f}    |    {pred[i]}')

acuracia = (pred == y).mean()
print(f'\nAcuracia no treino: {acuracia:.2%}')

Funcionario | anos | satisf | real | prob(saiu) | previsto
------------------------------------------------------------
     0      |  1   |   1    |  1   |   0.842    |    1
     1      |  2   |   1    |  1   |   0.842    |    1
     2      |  1   |   2    |  1   |   0.842    |    1
     3      |  5   |   3    |  0   |   0.158    |    0
     4      |  6   |   4    |  0   |   0.158    |    0
     5      |  8   |   3    |  0   |   0.158    |    0

Acuracia no treino: 100.00%


---
# Seção 2 - Mesmo exemplo com a biblioteca `xgboost`

Agora repetimos exatamente o mesmo problema, mas deixando a biblioteca oficial fazer todas as contas. O objetivo é:

1. ver como é **simples** usar a lib;
2. mapear cada hiperparâmetro de volta para a matemática da Seção 1;
3. confirmar que os resultados fazem sentido com o que implementamos à mão.

## 2.1 - Instalação / import

Se o `xgboost` ainda não estiver instalado no ambiente, rode no terminal:

```bash
uv pip install xgboost
```

(ou `pip install xgboost`). Depois, o import abaixo deve funcionar.

In [11]:
import xgboost as xgb
print('Versao do xgboost:', xgb.__version__)

Versao do xgboost: 3.4.1


## 2.2 - Treinando com a mesma configuração

Escolhemos os hiperparâmetros para **espelhar** a nossa implementação da Seção 1:

| Nosso código (Seção 1) | Parâmetro no `xgboost` | Significado |
| --- | --- | --- |
| `n_arvores=10` | `n_estimators=10` | número de árvores (iterações de boosting) |
| `learning_rate=0.3` | `learning_rate=0.3` | encolhimento $\eta$ de cada árvore |
| árvore de 1 corte | `max_depth=1` | profundidade - stump como o nosso |
| `lam=1.0` | `reg_lambda=1.0` | regularização L2 nas folhas |
| `gamma=0.0` | `gamma=0.0` | ganho mínimo para abrir um split |

Também desligamos subsample (`subsample=1.0`, `colsample_bytree=1.0`) para que, assim como no nosso modelo, cada árvore use **todos** os dados e todas as features.

**Dois detalhes que a nossa implementação do zero NÃO tinha e que precisamos ajustar para bater com ela:**

- `min_child_weight=0`: por padrão o XGBoost exige que a soma dos hessianos ($\sum h_i$) de cada folha seja pelo menos 1. No nosso dataset minúsculo, cada observação tem $h=0.25$ na 1ª iteração, então uma folha com 3 pessoas soma $0.75 < 1$ - **todos os splits seriam rejeitados e o modelo não aprenderia nada** (ficaria preso em 0.5 para todos!). Colocando `min_child_weight=0` destravamos isso, igual ao nosso código que não tinha essa restrição.
- `base_score=0.5`: fixamos a probabilidade inicial em 0.5 (log-odds = 0), exatamente o nosso base score calculado na Seção 1.2.

> Essa "pegadinha" do `min_child_weight` é um ótimo exemplo de por que entender a matemática (Seção 1) ajuda a não se perder com a biblioteca.

In [12]:
modelo_xgb = xgb.XGBClassifier(
    n_estimators=10,
    learning_rate=0.3,
    max_depth=1,          # stump, igual a nossa arvore de 1 corte
    reg_lambda=1.0,       # lambda (L2)
    gamma=0.0,            # ganho minimo p/ split
    min_child_weight=0,   # destrava splits no dataset minusculo (ver texto acima)
    base_score=0.5,       # prob. inicial 0.5 = log-odds 0, igual a Secao 1.2
    subsample=1.0,        # usa todas as linhas
    colsample_bytree=1.0, # usa todas as features
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
)

modelo_xgb.fit(X, y)
print('Modelo XGBoost treinado!')

Modelo XGBoost treinado!


## 2.3 - Previsões da biblioteca

Pegamos as probabilidades da classe positiva (coluna 1 de `predict_proba`) e comparamos com o padrão esperado.

In [13]:
proba_xgb = modelo_xgb.predict_proba(X)[:, 1]
pred_xgb = modelo_xgb.predict(X)

print('Funcionario | real | prob(saiu) lib | previsto lib')
print('-' * 52)
for i in range(len(X)):
    print(f'     {i}      |  {int(y[i])}   |     {proba_xgb[i]:.3f}      |      {pred_xgb[i]}')

print(f'\nAcuracia XGBoost (lib): {(pred_xgb == y).mean():.2%}')

Funcionario | real | prob(saiu) lib | previsto lib
----------------------------------------------------
     0      |  1   |     0.842      |      1
     1      |  1   |     0.842      |      1
     2      |  1   |     0.842      |      1
     3      |  0   |     0.158      |      0
     4      |  0   |     0.158      |      0
     5      |  0   |     0.158      |      0

Acuracia XGBoost (lib): 100.00%


## 2.4 - Comparando: do zero vs biblioteca

Como configuramos a biblioteca para espelhar fielmente a nossa implementação (mesmo base score, mesmo $\lambda$, mesma profundidade, sem a trava do `min_child_weight`), as probabilidades devem ficar **praticamente idênticas**. Essa é a melhor confirmação possível de que entendemos o algoritmo: a conta que fizemos na mão produz o mesmo resultado que a implementação profissional do XGBoost.

In [14]:
print('Func | real | prob DO ZERO | prob LIB  | diferenca')
print('-' * 55)
for i in range(len(X)):
    diff = abs(proba[i] - proba_xgb[i])
    print(f'  {i}  |  {int(y[i])}   |    {proba[i]:.3f}     |   {proba_xgb[i]:.3f}  |   {diff:.3f}')

max_diff = max(abs(proba[i] - proba_xgb[i]) for i in range(len(X)))
print(f'\nDiferenca maxima entre as duas implementacoes: {max_diff:.4f}')
print('Praticamente iguais! A matematica que fizemos na mao reproduz a biblioteca.')

Func | real | prob DO ZERO | prob LIB  | diferenca
-------------------------------------------------------
  0  |  1   |    0.842     |   0.842  |   0.000
  1  |  1   |    0.842     |   0.842  |   0.000
  2  |  1   |    0.842     |   0.842  |   0.000
  3  |  0   |    0.158     |   0.158  |   0.000
  4  |  0   |    0.158     |   0.158  |   0.000
  5  |  0   |    0.158     |   0.158  |   0.000

Diferenca maxima entre as duas implementacoes: 0.0000
Praticamente iguais! A matematica que fizemos na mao reproduz a biblioteca.


## 2.5 - Importância das features

Um bônus que a biblioteca entrega de graça: quanto cada feature contribuiu. No nosso exemplo, o corte em `anos_empresa <= 3.5` já separa perfeitamente as classes (foi exatamente o split que a Seção 1.4 encontrou). Por isso a árvore nunca precisa usar `satisfacao`, e a importância fica concentrada em `anos_empresa`. É o mesmo comportamento da nossa implementação do zero, que também escolheu `anos_empresa` como único corte.

In [15]:
importancias = modelo_xgb.feature_importances_
for nome, imp in zip(feature_names, importancias):
    print(f'{nome:15s}: {imp:.4f}')

anos_empresa   : 1.0000
satisfacao     : 0.0000


---
## Resumo do que aprendemos

**Seção 1 (do zero)** mostrou que o XGBoost, por dentro, é um loop simples:

1. **Base score** = palpite inicial em log-odds.
2. **Gradiente/Hessiano** = como medir o erro de cada ponto (derivadas da log-loss).
3. **Ganho do split** = como a árvore escolhe onde cortar.
4. **Valor da folha** $w^* = -G/(H+\lambda)$ = o ajuste ótimo de cada folha.
5. **Atualização** $\hat{y}_t = \hat{y}_{t-1} + \eta f_t$ = somar a árvore nova, devagar.

**Seção 2 (biblioteca)** mostrou que tudo isso vira três linhas: criar, `fit`, `predict`. Cada hiperparâmetro que você ajusta (`learning_rate`, `max_depth`, `reg_lambda`, `gamma`...) corresponde diretamente a uma peça da matemática que implementamos na mão.

Entender a Seção 1 é o que te dá intuição para **ajustar com consciência** os parâmetros da Seção 2.